# Instruction tuning GPT2.

Models that have had `sft` applied are called `Instruct` models. I will show how that works be instruction tuning gpt2.

**This is a little more advance than prompting**

### I Will do what I did in the notebook titled `gpt2-base-model.ipynb` just in case it was skipped. I will show the reason models are need to have SFT

SFT is the same as instruction tuning. Instruction tuning is not the same as fine-tuning. Fine-tuning is when you update ALL of the weights on a pre-trained model. This should NEVER be done.

In [1]:
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

prompts = (
    "today I was walking to the store and asked someone for a dollar",
    "I happened to forget my wallet and stumbled down the road",
    "I asked a man sitting if he could `spare any change`",
    "he looked at me grim: `you got codder, do you not`?",
    "The man must have been 60, wearing a shirt protesting the war",
)

inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.pad_token_id,
)

for i, prompt in enumerate(prompts):
    prompt_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[i][prompt_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    print(f"==== Prompt {i} ====")
    print(prompt)
    print(f"==== Base model reply {i} ====")
    print(response)
    print()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


==== Prompt 0 ====
today I was walking to the store and asked someone for a dollar
==== Base model reply 0 ====
 of coffee. She asked what I had. I told her I was buying a bottle of coffee and that she wanted to know the price. She asked if I had a bottle of coffee. I told her I had a bottle of coffee. She told

==== Prompt 1 ====
I happened to forget my wallet and stumbled down the road
==== Base model reply 1 ====
 when my phone went off the charger. I was very relieved. I had no idea how many days after that I could have spent on the phone. I was also very worried about how this would affect my other services.

I also had to

==== Prompt 2 ====
I asked a man sitting if he could `spare any change`
==== Base model reply 2 ====
 (i.e., if he knew the name of the person who stole the money, then he would not have to ask the police).

Then, he said, the woman who was trying to help him, she went back to the bank

==== Prompt 3 ====
he looked at me grim: `you got codder, do you not`?
====

## Why SFT is important.

The about output is just predicting the next sequence of token based on the tokens is was given. Not very useful without post-training.

In [2]:
import gc

del tokenizer, model, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

### Why this output is important.

Without SFT, the model is pretty much just auto complete. As more data is added and other things (I will get to) are added, the model does become better. Models need instruction-fine tuning to be able, to be simple, to mimic chat data.



# SFT GPT2

I will perform SFT on GPT2 using dataset name `databricks/databricks-dolly-15k` from hugging face hub. This is going to make it so the LLM has better responses.

In [3]:
from huggingface_hub import login
from datasets import load_dataset

login()

ds = load_dataset("vietgpt/databricks_dolly15k_en")['train']
ds = ds.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = ds["train"], ds["test"]

README.md:   0%|          | 0.00/362 [00:00<?, ?B/s]

data/train-00000-of-00001-54936bb8393acf(…): reconstructing file:   0%|          |  0.00B / 7.32MB            

data/train-00000-of-00001-54936bb8393acf(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15014 [00:00<?, ? examples/s]

### Format the chat template so the responses in the dataset are given to the model

This formatting approach is heavily influenced by Philipp Schmid. He is probably one of the greatest modern ML Engineers. [source](https://github.com/philschmid/deep-learning-pytorch-huggingface/blob/main/training/instruction-tune-llama-2-int4.ipynb)

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}"
    "### Instruction:\n{{ message['content'] }}\n\n### Response:\n"
    "{% elif message['role'] == 'assistant' %}"
    "{% generation %}{{ message['content'] }}{{ eos_token }}{% endgeneration %}"
    "{% endif %}"
    "{% endfor %}"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
tokenizer.chat_template = CHAT_TEMPLATE

print(tokenizer.apply_chat_template(train_ds[0]["messages"], tokenize=False))



### Instruction:
Give me a list of some different summer holidays that occur in the United States

### Response:
Some summer Holidays include Memorial Day, Fourth of July, Juneteenth and Labor Day<|endoftext|>


In [5]:
from peft import LoraConfig

peft_config = LoraConfig(
        lora_alpha=8,
        lora_dropout=0.05,
        r=6,
        bias="none",
        target_modules="all-linear",
        task_type="CAUSAL_LM",
)

In [7]:
!pip install -qqq trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.4 MB/s eta 0:00:00


In [9]:
!
!pip install --upgrade torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 50.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [10]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="gpt2-instruct-chat",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=3,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    max_length=256,
    learning_rate=2e-5,
    lr_scheduler_type="linear",
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=250,
    bf16=True,
    report_to="none",
)

model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.model.print_trainable_parameters()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/14263 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1249 > 1024). Running this sequence through the model will result in indexing errors


Building labels for train dataset:   0%|          | 0/14263 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/751 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/751 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/751 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/751 [00:00<?, ? examples/s]

trainable params: 884,736 || all params: 125,324,544 || trainable%: 0.7060


## Train The model

This should be done a a GPU that is not a consumer grade gpu.

In [11]:
trainer.train()

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,3.648240,3.331245,3.392330,53428.000000,0.432450
200,3.325588,3.132333,3.350839,108135.000000,0.435602
300,3.213313,3.000878,3.217227,157081.000000,0.443384
400,3.160660,2.924346,3.097344,209567.000000,0.450083
500,3.100080,2.881721,3.024127,263539.000000,0.453303
600,3.081939,2.844583,2.973593,318939.000000,0.457925
700,3.106758,2.821324,2.942299,371707.000000,0.460863
800,3.109109,2.808141,2.943082,425826.000000,0.461204
900,2.950253,2.796320,2.924913,480605.000000,0.461607
1000,2.939943,2.789059,2.921918,531274.000000,0.463043


TrainOutput(global_step=7132, training_loss=2.928690801225749, metrics={'train_runtime': 4841.0773, 'train_samples_per_second': 5.892, 'train_steps_per_second': 1.473, 'total_flos': 3278150591256576.0, 'train_loss': 2.928690801225749, 'epoch': 2.0})

### Not Good results.

### Save The Models

In [12]:
output_dir = "gpt2-instruct-chat"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

('gpt2-instruct-chat/tokenizer_config.json',
 'gpt2-instruct-chat/chat_template.jinja',
 'gpt2-instruct-chat/tokenizer.json')

In [13]:
import gc

gc.collect()
torch.cuda.empty_cache()

## Compare the results from the SFT Model

In [15]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

ADAPTER_PATH = "/content/gpt2-instruct-chat"
BASE_MODEL = "gpt2"
device = "cuda:0"

gen_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
gen_tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL).eval()

tuned_model = (
    PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(BASE_MODEL),
        ADAPTER_PATH,
    )
    .merge_and_unload()
    .eval()
    .to(device)
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

## Outputs from SFT model

In [16]:
instructions = (
    "Explain what supervised fine-tuning is.",
    "Give me three ideas for a weekend trip.",
    "What is the capital of Australia?",
)

for instruction in instructions:
    text = gen_tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction}],
        tokenize=False,
    )
    inputs = gen_tokenizer(text, return_tensors="pt").to(device)

    outputs = tuned_model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        eos_token_id=gen_tokenizer.eos_token_id,
        pad_token_id=gen_tokenizer.eos_token_id,
    )
    reply = gen_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :],
        skip_special_tokens=True,
    )

    print("==== Instruction ====")
    print(instruction)
    print("==== SFT reply ====")
    print(reply)
    print()

==== Instruction ====
Explain what supervised fine-tuning is.
==== SFT reply ====
There are three types of fines that can be levied against a person for speeding or driving without insurance, and these vary by state and city in each jurisdiction.

==== Instruction ====
Give me three ideas for a weekend trip.
==== SFT reply ====
1) A bikepacking excursion with friends and family in the mountains of Utah, to visit their favorite places like Glacier National Park or Mt. Hood Mountain Resort (both are great hiking destinations). 2] This could be fun at first but then you'd have more issues going on vacation that can get worse over time! 3-) Try taking some water out after each hike so as not leave too much behind from your previous adventure; this would make it easier to find something else while still enjoying nature's beauty during summer months

==== Instruction ====
What is the capital of Australia?
==== SFT reply ====
Australia has a population that ranges from 6,500 to 16 million. Th